### First we fix the .json with the samples to follow the new exact_match logic

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

def get_entity_key(ent):
    key_dict = ent.get('key', {})
    return (key_dict.get('label'), key_dict.get('start_char'), key_dict.get('end_char'))

def _comparison_text(ent):
    key_dict = ent.get('key', {})
    return str(ent.get('normalized_text') or key_dict.get('text') or "")

def fix_ner_key(ent):
    attrs = ent.get('extra', {}).get('attrs', {})
    if attrs and 'key' in ent:
        ent['key']['text'] = attrs.get('aymurai_alt_text', ent['key'].get('text'))
        ent['key']['start_char'] = attrs.get('aymurai_alt_start_char', ent['key'].get('start_char'))
        ent['key']['end_char'] = attrs.get('aymurai_alt_end_char', ent['key'].get('end_char'))
    return ent

def recompute_comparison(ner_entities, lx_entities):
    ner_map = defaultdict(list)
    lx_map = defaultdict(list)
    for ent in ner_entities:
        ner_map[get_entity_key(ent)].append(ent)
    for ent in lx_entities:
        lx_map[get_entity_key(ent)].append(ent)

    exact_match, remaining_ner, remaining_lx = [], [], []
    all_exact_keys = set(ner_map.keys()) | set(lx_map.keys())
    
    for key in all_exact_keys:
        ner_items, lx_items = ner_map.get(key, []), lx_map.get(key, [])
        n_match = min(len(ner_items), len(lx_items))
        exact_match.extend(ner_items[:n_match])
        if len(ner_items) > n_match: remaining_ner.extend(ner_items[n_match:])
        if len(lx_items) > n_match: remaining_lx.extend(lx_items[n_match:])

    partial_match, matched_idx_lx, matched_idx_ner = [], set(), set()
    for i, ner_ent in enumerate(remaining_ner):
        for j, lx_ent in enumerate(remaining_lx):
            if j in matched_idx_lx: continue
            ner_text, lx_text = _comparison_text(ner_ent), _comparison_text(lx_ent)
            if (ner_ent.get('label') == lx_ent.get('label') and lx_text and lx_text in ner_text):
                partial_match.append(ner_ent)
                matched_idx_ner.add(i)
                matched_idx_lx.add(j)
                break

    only_in_ner = [e for i, e in enumerate(remaining_ner) if i not in matched_idx_ner]
    only_in_lx = [e for i, e in enumerate(remaining_lx) if i not in matched_idx_lx]

    if not remaining_ner and not remaining_lx:
        status = "exact_match"
    elif partial_match and not exact_match and not only_in_ner and not only_in_lx:
        status = "partial_match"
    elif only_in_ner and not only_in_lx and not exact_match and not partial_match:
        status = "ner_only"
    elif only_in_lx and not only_in_ner and not exact_match and not partial_match:
        status = "langextract_only"
    else:
        status = "mixed"

    return {
        "status": status,
        "exact_match": exact_match,
        "partial_match": partial_match,
        "only_in_ner": only_in_ner,
        "only_in_langextract": only_in_lx
    }

In [ ]:
input_file = 'public/predictions_raw/samples_raw.json'
output_file = 'curated/predictions_raw/public_samples_raw_curated.json'

output_path = Path(output_file)
output_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Loading data from {input_file}...")
with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

total_samples = len(data)
print(f"Processing {total_samples} samples...")

for idx, item in enumerate(data, 1):
    # Tracking logs
    if idx % 10 == 0 or idx == total_samples:
        print(f"Progress: {idx}/{total_samples} ({(idx/total_samples)*100:.1f}%)")

    # Core logic
    item['ner_predictions'] = [fix_ner_key(ent) for ent in item['ner_predictions']]
    
    new_comparison = recompute_comparison(
        item['ner_predictions'], 
        item['langextract_predictions']
    )
    
    item['comparison'] = new_comparison

print(f"Saving curated data to {output_file}...")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Curation process completed successfully.")

In [ ]:
input_file = 'restricted/predictions_raw/samples_raw.json'
output_file = 'curated/predictions_raw/restricted_samples_raw_curated.json'

output_path = Path(output_file)
output_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Loading data from {input_file}...")
with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

total_samples = len(data)
print(f"Processing {total_samples} samples...")

for idx, item in enumerate(data, 1):
    # Tracking logs
    if idx % 10 == 0 or idx == total_samples:
        print(f"Progress: {idx}/{total_samples} ({(idx/total_samples)*100:.1f}%)")

    # Core logic
    item['ner_predictions'] = [fix_ner_key(ent) for ent in item['ner_predictions']]
    
    new_comparison = recompute_comparison(
        item['ner_predictions'], 
        item['langextract_predictions']
    )
    
    item['comparison'] = new_comparison

print(f"Saving curated data to {output_file}...")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Curation process completed successfully.")

### Now we try to export to labelstudio this samples and to make the train candidates.

In [ ]:
from aymurai.experiments.ner_langextract_alignment.labelstudio_export import export_labelstudio_tasks
from aymurai.experiments.ner_langextract_alignment.labelstudio_import import consolidate_datasets, load_labelstudio_tasks, parse_labelstudio_decisions

In [ ]:
def write_jsonl(path: Path, data: list[dict]):
    with open(path, "w", encoding="utf-8") as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

class PipelineConfig:
    def __init__(self):
        self.input_base_dir = Path("curated/predictions_raw")
        self.ls_export_dir = Path("curated/labelstudio_export")
        self.train_candidates_path = Path("curated/train_candidates.jsonl")
        self.review_required_path = Path("curated/review_required.jsonl")
        self.qa_sample_rate = 0.1
        self.qa_seed = 42
        self.import_annotations_path = None

config = PipelineConfig()

def run_multi_file_pipeline():
    all_samples = []
    json_files = list(config.input_base_dir.rglob("*.json"))
    
    if not json_files:
        print(f"No JSON files found in {config.input_base_dir}")
        return

    for file_path in json_files:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                content = json.load(f)
                if isinstance(content, list):
                    all_samples.extend(content)
                else:
                    all_samples.append(content)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    if not all_samples:
        return

    config.ls_export_dir.mkdir(parents=True, exist_ok=True)

    ls_paths = export_labelstudio_tasks(
        samples=all_samples,
        export_dir=config.ls_export_dir,
        qa_sample_rate=config.qa_sample_rate,
        qa_seed=config.qa_seed,
    )

    decisions = {}
    if config.import_annotations_path and Path(config.import_annotations_path).exists():
        tasks = load_labelstudio_tasks(Path(config.import_annotations_path))
        decisions = parse_labelstudio_decisions(tasks)

    train_candidates, review_required = consolidate_datasets(all_samples, decisions)

    write_jsonl(config.train_candidates_path, train_candidates)
    write_jsonl(config.review_required_path, review_required)

    print(f"Processed files: {len(json_files)}")
    print(f"Total samples: {len(all_samples)}")
    print(f"Train candidates: {len(train_candidates)}")
    print(f"Review required: {len(review_required)}")


In [ ]:
if __name__ == "__main__":
    run_multi_file_pipeline()

In [ ]:
jsonp={"sample_id": "7b7caff9-21c7-5cb0-b403-80beecc4600a-16", "document_id": "7b7caff9-21c7-5cb0-b403-80beecc4600a", "paragraph_id": "16", "source_path": "/resources/data/pjn/Consejo-de-la-Magistratura/Resoluciones-de-Presidencia---Año-2023/2023/02-06-2023-000000_El-expediente-N-482022--caratulado-M.D.-c-Dra.-Díaz-Cordero-Agustina-Juzgado-Nacional-de-Prim.-Inst.-e.pdf", "text": "HORACIO ROSATTI", "ner_predictions": [{"label": "PER", "start_char": 0, "end_char": 15, "text": "HORACIO ROSATTI", "source": "ner", "raw_label": "PER", "normalized_text": "horacio rosatti", "extra": {"attrs": {"aymurai_label": "PER", "aymurai_label_subclass": [], "aymurai_alt_text": "HORACIO ROSATTI", "aymurai_alt_start_char": 0, "aymurai_alt_end_char": 15, "aymurai_method": "ner/flair", "aymurai_score": 0.7461150586605072, "aymurai_label_instance": None, "aymurai_disambiguation": None, "aymurai_anonymize": None, "canonical_entity_id": None}, "original_text": "HORACIO ROSATTI"}, "key": {"label": "PER", "start_char": 0, "end_char": 15, "text": "HORACIO ROSATTI"}}], "langextract_predictions": [{"label": "PER", "start_char": 0, "end_char": 15, "text": "HORACIO ROSATTI", "source": "langextract", "raw_label": "PER", "normalized_text": "horacio rosatti", "extra": {"raw_class": "PER", "raw_text": "HORACIO ROSATTI", "alignment_status": "match_exact"}, "key": {"label": "PER", "start_char": 0, "end_char": 15, "text": "HORACIO ROSATTI"}}], "comparison": {"status": "exact_match", "exact_match": [{"label": "PER", "start_char": 0, "end_char": 15, "text": "HORACIO ROSATTI", "source": "ner", "raw_label": "PER", "normalized_text": "horacio rosatti", "extra": {"attrs": {"aymurai_label": "PER", "aymurai_label_subclass": [], "aymurai_alt_text": "HORACIO ROSATTI", "aymurai_alt_start_char": 0, "aymurai_alt_end_char": 15, "aymurai_method": "ner/flair", "aymurai_score": 0.7461150586605072, "aymurai_label_instance": None, "aymurai_disambiguation": None, "aymurai_anonymize": None, "canonical_entity_id": None}, "original_text": "HORACIO ROSATTI"}, "key": {"label": "PER", "start_char": 0, "end_char": 15, "text": "HORACIO ROSATTI"}}], "partial_match": [], "only_in_ner": [], "only_in_langextract": []}, "quality_flags": [], "timestamps": {"processed_at": "2026-02-22T08:47:11.809838+00:00"}, "latency_ms": {"ner": 1456.0423650000303, "langextract": 2784.2079840002043}, "final_entities": [{"label": "PER", "start_char": 0, "end_char": 15, "text": "HORACIO ROSATTI", "source": "ner", "raw_label": "PER", "normalized_text": "horacio rosatti", "extra": {"attrs": {"aymurai_label": "PER", "aymurai_label_subclass": [], "aymurai_alt_text": "HORACIO ROSATTI", "aymurai_alt_start_char": 0, "aymurai_alt_end_char": 15, "aymurai_method": "ner/flair", "aymurai_score": 0.7461150586605072, "aymurai_label_instance": None, "aymurai_disambiguation": None, "aymurai_anonymize": None, "canonical_entity_id": None}, "original_text": "HORACIO ROSATTI"}, "key": {"label": "PER", "start_char": 0, "end_char": 15, "text": "HORACIO ROSATTI"}}], "final_decision": "auto_exact_match"}

In [ ]:
%load_ext rich

In [ ]:
jsonp